<a href="https://colab.research.google.com/github/hamedhossam22/repo-Multimodal-Review-Consistency-Detection/blob/main/Hamed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import Transfer Learning Libraries

from tensorflow.keras.applications import ResNet50

from tensorflow.keras.applications.resnet50 import preprocess_input

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (

    Dense,
    GlobalAveragePooling2D,
    Dropout

)

In [ ]:
# Preprocess Images for ResNet50

X_train_resnet = preprocess_input(
    X_train_img * 255
)

X_test_resnet = preprocess_input(
    X_test_img * 255
)

print("ResNet preprocessing completed successfully!")

In [ ]:
# Load ResNet50 Base Model

base_model = ResNet50(

    weights='imagenet',

    include_top=False,

    input_shape=(128,128,3)
)

print("ResNet50 base model loaded successfully!")

In [ ]:
# Freeze ResNet50 Layers

for layer in base_model.layers:

    layer.trainable = False

print("ResNet50 layers frozen successfully!")

In [ ]:
# Build ResNet50 Transfer Learning Model

x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    128,
    activation='relu'
)(x)

x = Dropout(0.5)(x)

output = Dense(
    1,
    activation='sigmoid'
)(x)

resnet_model = Model(
    inputs=base_model.input,
    outputs=output
)

print("ResNet50 transfer model created successfully!")

In [ ]:
# Compile ResNet50 Model

resnet_model.compile(

    optimizer='adam',

    loss='binary_crossentropy',

    metrics=['accuracy']
)

print("ResNet50 model compiled successfully!")

In [ ]:
history_resnet = resnet_model.fit(

    X_train_img,
    y_train_img,

    validation_data=(X_test_img, y_test_img),

    epochs=10,

    batch_size=16,
    callbacks=[early_stopping]

In [ ]:
# ResNet Accuracy Curves

plt.figure(figsize=(8,5))

plt.plot(
    history_resnet.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history_resnet.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title("ResNet50 Accuracy Curves")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.show()

In [ ]:
# ResNet Loss Curves

plt.figure(figsize=(8,5))

plt.plot(
    history_resnet.history['loss'],
    label='Training Loss'
)

plt.plot(
    history_resnet.history['val_loss'],
    label='Validation Loss'
)

plt.title("ResNet50 Loss Curves")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.show()

In [ ]:
# Evaluate ResNet50 Model

resnet_loss, resnet_accuracy = resnet_model.evaluate(

    X_test_resnet,
    y_test_img
)

print("ResNet50 Test Loss:", resnet_loss)
print("ResNet50 Test Accuracy:", resnet_accuracy)

In [ ]:
# Generate ResNet50 Predictions

resnet_predictions = resnet_model.predict(
    X_test_resnet
)

resnet_predictions = (
    resnet_predictions > 0.5
).astype(int)

In [ ]:
# ResNet50 Classification Report

print(

    classification_report(
        y_test_img,
        resnet_predictions
    )
)